In [2]:
import sys
sys.path.append('../src')
sys.path.append('/shared_data0/weiqiuy/api_keys')
import json
import os

from data_utils import load_jsonl
from text_utils import text2json
from eval_utils import compute_precision_from_results, compute_recall_from_results

# technontech2claims

In [3]:
model_name = 'gpt-4o-mini'
debug = False
threshold = 0.9
task_name = 'technontech2claims'

In [18]:
# load_paths = {
#     # f'/shared_data0/weiqiuy/nsf-awards/results/{task_name}/mistral-7b-instruct-v0.3_preds/gen_only_results.jsonl': 'Mistral',
#     # f'/shared_data0/weiqiuy/nsf-awards/results/{task_name}/lora_model_mistral-7b-instruct-v0.3.technontech2claims_instruct_user_assistant.r128_la64_lr1e-05_e3_s-1_wu100_bs2_schlinear_optadamw_8bit_emb_lm_preds/gen_only_results.jsonl': 'Mistral-ft',
#     # f'/shared_data0/weiqiuy/nsf-awards/results/{task_name}/Qwen2.5-7B-Instruct_preds/gen_only_results.jsonl': 'Qwen',
#     # f'/shared_data0/weiqiuy/nsf-awards/results/{task_name}/lora_model_Qwen2.5-7B-Instruct.technontech2claims_instruct.r128_la64_lr1e-05_e3_s-1_wu100_bs2_schlinear_optadamw_8bit_emb_lm_preds/gen_only_results.jsonl': 'Qwen-ft',
#     # f'/shared_data0/weiqiuy/nsf-awards/results/technontech2claimsip/mistral-7b-instruct-v0.3_preds/gen_only_results.jsonl': 'Mistral-comb',
#     # f'/shared_data0/weiqiuy/nsf-awards/results/technontech2claimsip/lora_model_mistral-7b-instruct-v0.3.technontech2claimsip_instruct_user_assistant.r128_la64_lr1e-05_e3_s-1_wu100_bs2_schlinear_optadamw_8bit_emb_lm_preds/gen_only_results.jsonl': 'Mistral-comb-ft',
#     # f'/shared_data0/weiqiuy/nsf-awards/results/technontech2claimsip/Qwen2.5-7B-Instruct_preds/gen_only_results.jsonl': 'Qwen-comb',
#     # f'/shared_data0/weiqiuy/nsf-awards/results/technontech2claimsip/lora_model_Qwen2.5-7B-Instruct.technontech2claimsip_instruct.r128_la64_lr1e-05_e3_s-1_wu100_bs2_schlinear_optadamw_8bit_emb_lm_preds/gen_only_results.jsonl': 'Qwen-comb-ft',
#     f'/shared_data0/weiqiuy/nsf-awards/results/{task_name}_debug/lora_model_mistral-7b-instruct-v0.3.text2claims_instruct_user_assistant.r128_la64_lr1e-05_e3_s-1_wu100_bs2_schlinear_optadamw_8bit_emb_lm_preds/gen_only_results.jsonl': 'Mistral-ft-text',
# }

In [24]:
load_dirs = {
    f'/shared_data0/weiqiuy/nsf-awards/results_gen_bsz1/{task_name}/mistral-7b-instruct-v0.3_preds': 'Mistral',
    f'/shared_data0/weiqiuy/nsf-awards/results_gen_bsz1/{task_name}/lora_model_mistral-7b-instruct-v0.3.text2claims_instruct_user_assistant.r128_la64_lr1e-05_e3_s-1_wu100_bs2_schlinear_optadamw_8bit_emb_lm_preds': 'Mistral-SciFy-MatSci',
    f'/shared_data0/weiqiuy/nsf-awards/results_gen_bsz1/{task_name}/lora_model_mistral-7b-instruct-v0.3.text2claims_instruct_user_assistant.matsci_and_20k.r128_la64_lr1e-05_e3_s-1_wu100_bs2_schlinear_optadamw_8bit_emb_lm_preds': 'Mistral-SciFy'
}

In [26]:
import os
import json
import math
import numpy as np
import re
from collections import defaultdict
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from data_utils import get_nsf_data_raw, get_nsf_data_matsci_and_20k_raw

# === Your existing functions ===
p_r_f_all = defaultdict(dict)

# li = 1
# for li in range(len(load_paths)):

#     load_path = load_paths[li]
# for load_path in load_paths:
for load_dir in load_dirs:

    # load_dir = os.path.dirname(load_path)
    print('load_dir', load_dir)

    new_data_path = os.path.join(load_dir, 'claims_extracted.jsonl')
    new_data = load_jsonl(new_data_path)

    filenames = os.listdir(load_dir)

    # precision

    # if 'technontech2claimsip' in load_dir or 'text2claims' in load_dir:
    pattern = re.compile(r'^claims_openai_requests_\d+_precision\.jsonl\.gpt-4o-mini_response\.jsonl$')
    # else:
    #     pattern = re.compile(r'^openai_requests_\d+\.jsonl\.gpt-4o-mini_response\.jsonl$')

    matched_files = [f for f in sorted(filenames) if pattern.match(f)]

    responses = []
    for filename in matched_files:
        responses.extend(load_jsonl(os.path.join(load_dir, filename)))


    threshold = 0.9
    precision_results = defaultdict(dict)
    for i in range(len(responses)):
        line = json.loads(responses[i]['line'])
        custom_id = line['custom_id']
        completion = line['response']['body']['choices'][0]['logprobs']['content'][0]['token'].strip().lower()
        logprob = line['response']['body']['choices'][0]['logprobs']['content'][0]['logprob']

        data_id, pred_claim_i, gold_claim_i = custom_id.split('.')
        data_id, pred_claim_i, gold_claim_i = int(data_id.split('_')[1]), \
            int(pred_claim_i.split('_')[1]), int(gold_claim_i.split('_')[1])
        # data_id, pred_claim_i, gold_claim_i

        precision_results[data_id][tuple([pred_claim_i, gold_claim_i])] = completion == 'yes' and math.exp(logprob) >= threshold

    # recall

    # if 'technontech2claimsip' in load_dir or 'text2claims' in load_dir:
    pattern = re.compile(r'^claims_openai_requests_\d+_recall\.jsonl\.gpt-4o-mini_response\.jsonl$')
    # else:
    #     pattern = re.compile(r'^openai_requests_\d+_recall\.jsonl\.gpt-4o-mini_response\.jsonl$')

    print('filenames', filenames)
    matched_files = [f for f in sorted(filenames) if pattern.match(f)]
    print(matched_files)

    responses = []
    for filename in matched_files:
        responses.extend(load_jsonl(os.path.join(load_dir, filename)))

    threshold = 0.9
    recall_results = defaultdict(dict)
    for i in range(len(responses)):
        line = json.loads(responses[i]['line'])
        custom_id = line['custom_id']
        completion = line['response']['body']['choices'][0]['logprobs']['content'][0]['token'].strip().lower()
        logprob = line['response']['body']['choices'][0]['logprobs']['content'][0]['logprob']

        data_id, pred_claim_i, gold_claim_i = custom_id.split('.')
        data_id, pred_claim_i, gold_claim_i = int(data_id.split('_')[1]), \
            int(pred_claim_i.split('_')[1]), int(gold_claim_i.split('_')[1])
        # data_id, pred_claim_i, gold_claim_i
        # print(completion, math.exp(logprob))
        # break
        recall_results[data_id][tuple([pred_claim_i, gold_claim_i])] = completion == 'yes' and math.exp(logprob) >= threshold



    # === Load your data/results ===
    # (Assuming load_paths, load_jsonl, train_dataset, new_data, precision_results, and recall_results are already defined)
    # For this example, we assume:
    #   - train_dataset['verifiable_claims'] is a list of lists of verifiable claims.
    #   - new_data is a list of dicts with key 'pred_claims'
    #   - precision_results and recall_results are dicts keyed by data instance id (integers)
    #   - load_dir is derived from one of the load_paths


    train_dataset, val_dataset, test_dataset = get_nsf_data_matsci_and_20k_raw('..') #get_nsf_data_raw('..')

    # === Set up TF-IDF on verifiable claims ===

    # Flatten verifiable claims if needed (each inner list is from one data instance)
    verifiable_claims_flat = [claim for claims_list in train_dataset['verifiable_claims'] \
                              for claim in claims_list]

    # Fit TF-IDF vectorizer on verifiable claims only
    vectorizer = TfidfVectorizer()
    vectorizer.fit(verifiable_claims_flat)

    # === Compute Non-filtered Precision and Recall ===

    nonfiltered_precisions = []
    nonfiltered_recalls = []

    # Here, we assume that the keys of precision_results and recall_results correspond to the data instance indices.
    for di in range(len(new_data)):
        nonfiltered_precisions.append(compute_precision_from_results(precision_results[di]))
        nonfiltered_recalls.append(compute_recall_from_results(recall_results[di]))

    # === Redundancy Removal on Predicted Claims ===

    # For each data instance, we determine which predicted claims are unique.
    unique_pred_indices = {}  # data instance id -> list of indices of unique predicted claims

    for di in tqdm(range(len(new_data)), desc="Computing unique predicted claims"):
        pred_claims = new_data[di]['pred_claims']
        if len(pred_claims) == 0:
            unique_pred_indices[di] = []
            continue

        # Transform predicted claims using the vectorizer (fitted on verifiable claims)
        pred_tfidf = vectorizer.transform(pred_claims)

        # Compute pairwise cosine similarity among predicted claims
        similarity_matrix = cosine_similarity(pred_tfidf, pred_tfidf)
        np.fill_diagonal(similarity_matrix, 0)  # ignore self-similarity

        # Define a threshold above which two claims are considered redundant
        redundancy_threshold = 0.9

        unique_indices = []
        # For each claim, if it is too similar to any claim already selected, mark it as redundant.
        for i in range(len(pred_claims)):
            is_redundant = False
            for j in unique_indices:
                if similarity_matrix[i, j] >= redundancy_threshold:
                    is_redundant = True
                    break
            if not is_redundant:
                unique_indices.append(i)
        unique_pred_indices[di] = unique_indices

    # === Filter Precision and Recall Results Based on Unique Predicted Claims ===

    filtered_precision_results = {}
    filtered_recall_results = {}

    for di in unique_pred_indices:
        unique_set = set(unique_pred_indices[di])
        # Filter precision results: keep only keys where the predicted claim index is in the unique set.
        if di in precision_results:
            filtered_precision_results[di] = { key: val 
                for key, val in precision_results[di].items() if key[0] in unique_set }
            # print(list(unique_set)[0])

        else:
            filtered_precision_results[di] = {}

        if di in recall_results:
            filtered_recall_results[di] = { key: val 
                for key, val in recall_results[di].items() if key[0] in unique_set }
        else:
            filtered_recall_results[di] = {}

    # === Compute Filtered Precision and Recall ===

    filtered_precisions = []
    filtered_recalls = []

    for di in range(len(new_data)):
        filtered_precisions.append(compute_precision_from_results(filtered_precision_results[di]))
        filtered_recalls.append(compute_recall_from_results(filtered_recall_results[di]))

    # === Compute F-scores (optional) ===

    nonfiltered_fscores = [
        2 * (p * r) / (p + r) if (p + r) > 0 else 0.0 
        for p, r in zip(nonfiltered_precisions, nonfiltered_recalls)
    ]

    filtered_fscores = [
        2 * (p * r) / (p + r) if (p + r) > 0 else 0.0 
        for p, r in zip(filtered_precisions, filtered_recalls)
    ]

    # === Print Mean Metrics for Comparison ===

    print("Non-filtered Mean Precision:", np.mean(nonfiltered_precisions))
    print("Non-filtered Mean Recall:", np.mean(nonfiltered_recalls))
    print("Non-filtered Mean F-score:", np.mean(nonfiltered_fscores))

    print("Filtered Mean Precision:", np.mean(filtered_precisions))
    print("Filtered Mean Recall:", np.mean(filtered_recalls))
    print("Filtered Mean F-score:", np.mean(filtered_fscores))
    
    p_r_f_all[load_dirs[load_dir]]['precisions'] = filtered_precisions
    p_r_f_all[load_dirs[load_dir]]['recalls'] = filtered_recalls
    p_r_f_all[load_dirs[load_dir]]['fscores'] = filtered_fscores


load_dir /shared_data0/weiqiuy/nsf-awards/results_gen_bsz1/technontech2claims/mistral-7b-instruct-v0.3_preds
filenames ['claims_extracted.jsonl', 'claims_openai_requests_0_precision.jsonl', 'claims_openai_requests_16_precision.jsonl.gpt-4o-mini_response.jsonl', 'claims_openai_requests_10_precision.jsonl', 'claims_openai_requests_8_precision.jsonl.gpt-4o-mini_response.jsonl', 'claims_openai_requests_9_recall.jsonl.gpt-4o-mini_response.jsonl', 'claims_openai_requests_20_precision.jsonl', 'claims_openai_requests_1_recall.jsonl.gpt-4o-mini_response.jsonl', 'claims_openai_requests_19_recall.jsonl', 'claims_openai_requests_14_precision.jsonl.gpt-4o-mini_response.jsonl', 'claims_openai_requests_5_recall.jsonl.gpt-4o-mini_response.jsonl', 'claims_openai_requests_13_recall.jsonl', 'claims_openai_requests_12_precision.jsonl', 'claims_openai_requests_2_precision.jsonl', 'claims_openai_requests_12_precision.jsonl.gpt-4o-mini_response.jsonl', 'claims_openai_requests_11_recall.jsonl', 'claims_openai

Casting the dataset:   0%|          | 0/8641 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/14682 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

...

Casting the dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

[concat_datasets_drop_mismatches] dropping columns: ['publications']
[concat_datasets_drop_mismatches] dropping columns: ['award_year', 'publications']
[concat_datasets_drop_mismatches] dropping columns: ['publications']


Computing unique predicted claims:   0%|          | 0/6000 [00:00<?, ?it/s]

Non-filtered Mean Precision: 0.7524007121427185
Non-filtered Mean Recall: 0.7354208929414812
Non-filtered Mean F-score: 0.7266331046399476
Filtered Mean Precision: 0.7526250825217284
Filtered Mean Recall: 0.7354000596081478
Filtered Mean F-score: 0.7268255602323946


In [27]:
results_path = f'../metrics/{task_name}_final_scores.json'
os.makedirs(os.path.dirname(results_path), exist_ok=True)
with open(results_path, 'wt') as output_file:
    json.dump(p_r_f_all, output_file, indent=4)

# technontech2ip

In [31]:
import re
from collections import defaultdict
import math

model_name = 'gpt-4o-mini'
debug = False
threshold = 0.9
task_name = 'technontech2ip'

load_dirs = {
    f'/shared_data0/weiqiuy/nsf-awards/results_gen_bsz1/{task_name}/mistral-7b-instruct-v0.3_preds': 'Mistral',
    f'/shared_data0/weiqiuy/nsf-awards/results_gen_bsz1/{task_name}/lora_model_mistral-7b-instruct-v0.3.text2ip_instruct_user_assistant.r128_la64_lr1e-05_e3_s-1_wu100_bs2_schlinear_optadamw_8bit_emb_lm_preds': 'Mistral-SciFy-MatSci',
    f'/shared_data0/weiqiuy/nsf-awards/results_gen_bsz1/{task_name}/lora_model_mistral-7b-instruct-v0.3.text2ip_instruct_user_assistant.matsci_and_20k.r128_la64_lr1e-05_e3_s-1_wu100_bs2_schlinear_optadamw_8bit_emb_lm_preds': 'Mistral-SciFy'
}

# load_paths = {
#     # f'/shared_data0/weiqiuy/nsf-awards/results/{task_name}/mistral-7b-instruct-v0.3_preds/gen_only_results.jsonl': 'Mistral',
#     # f'/shared_data0/weiqiuy/nsf-awards/results/{task_name}/lora_model_mistral-7b-instruct-v0.3.technontech2ip_instruct_user_assistant.r128_la64_lr1e-05_e3_s-1_wu100_bs2_schlinear_optadamw_8bit_emb_lm_preds/gen_only_results.jsonl': 'Mistral-ft',
#     # f'/shared_data0/weiqiuy/nsf-awards/results/{task_name}/Qwen2.5-7B-Instruct_preds/gen_only_results.jsonl': 'Qwen',
#     # f'/shared_data0/weiqiuy/nsf-awards/results/{task_name}/lora_model_Qwen2.5-7B-Instruct.technontech2ip_instruct.r128_la64_lr1e-05_e3_s-1_wu100_bs2_schlinear_optadamw_8bit_emb_lm_preds/gen_only_results.jsonl': 'Qwen-ft',
#     # f'/shared_data0/weiqiuy/nsf-awards/results/technontech2claimsip/mistral-7b-instruct-v0.3_preds/gen_only_results.jsonl': 'Mistral-comb',
#     # f'/shared_data0/weiqiuy/nsf-awards/results/technontech2claimsip/lora_model_mistral-7b-instruct-v0.3.technontech2claimsip_instruct_user_assistant.r128_la64_lr1e-05_e3_s-1_wu100_bs2_schlinear_optadamw_8bit_emb_lm_preds/gen_only_results.jsonl': 'Mistral-comb-ft',
#     # f'/shared_data0/weiqiuy/nsf-awards/results/technontech2claimsip/Qwen2.5-7B-Instruct_preds/gen_only_results.jsonl': 'Qwen-comb',
#     # f'/shared_data0/weiqiuy/nsf-awards/results/technontech2claimsip/lora_model_Qwen2.5-7B-Instruct.technontech2claimsip_instruct.r128_la64_lr1e-05_e3_s-1_wu100_bs2_schlinear_optadamw_8bit_emb_lm_preds/gen_only_results.jsonl': 'Qwen-comb-ft'
#     f'/shared_data0/weiqiuy/nsf-awards/results/{task_name}_debug/lora_model_mistral-7b-instruct-v0.3.text2ip_instruct_user_assistant.r128_la64_lr1e-05_e3_s-1_wu100_bs2_schlinear_optadamw_8bit_emb_lm_preds/gen_only_results.jsonl': 'Mistral-ft-text',
# }

# load_paths = [
#     f'/shared_data0/weiqiuy/nsf-awards/results/{task_name}/mistral-7b-instruct-v0.3_preds/gen_only_results.jsonl',
#     f'/shared_data0/weiqiuy/nsf-awards/results/{task_name}/lora_model_mistral-7b-instruct-v0.3.technontech2ip_instruct_user_assistant.r128_la64_lr1e-05_e3_s-1_wu100_bs2_schlinear_optadamw_8bit_emb_lm_preds/gen_only_results.jsonl',
#     f'/shared_data0/weiqiuy/nsf-awards/results/{task_name}/Qwen2.5-7B-Instruct_preds/gen_only_results.jsonl',
#     f'/shared_data0/weiqiuy/nsf-awards/results/{task_name}/lora_model_Qwen2.5-7B-Instruct.technontech2ip_instruct.r128_la64_lr1e-05_e3_s-1_wu100_bs2_schlinear_optadamw_8bit_emb_lm_preds/gen_only_results.jsonl',
# ]

import os
import json
import math
import numpy as np
import re
from collections import defaultdict
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from data_utils import get_nsf_data_raw

# === Your existing functions ===
p_r_f_all = defaultdict(dict)

# li = 1
# for li in range(len(load_paths)):

    # load_path = load_paths[li]
for load_dir in load_dirs:
    # load_dir = os.path.dirname(load_path)
    print('load_dir', load_dir)

    new_data_path = os.path.join(load_dir, 'claims_extracted.jsonl')
    new_data = load_jsonl(new_data_path)

    filenames = os.listdir(load_dir)

    # precision

    # if 'technontech2claimsip' in load_dir or 'text2ip' in load_dir:
    pattern = re.compile(r'^ips_openai_requests_\d+_precision\.jsonl\.gpt-4o-mini_response\.jsonl$')
    # else:
    #     pattern = re.compile(r'^openai_requests_\d+\.jsonl\.gpt-4o-mini_response\.jsonl$')

    matched_files = [f for f in sorted(filenames) if pattern.match(f)]

    responses = []
    for filename in matched_files:
        responses.extend(load_jsonl(os.path.join(load_dir, filename)))


    threshold = 0.9
    precision_results = defaultdict(dict)
    for i in range(len(responses)):
        line = json.loads(responses[i]['line'])
        custom_id = line['custom_id']
        completion = line['response']['body']['choices'][0]['logprobs']['content'][0]['token'].strip().lower()
        logprob = line['response']['body']['choices'][0]['logprobs']['content'][0]['logprob']

        data_id, pred_claim_i, gold_claim_i = custom_id.split('.')
        data_id, pred_claim_i, gold_claim_i = int(data_id.split('_')[1]), \
            int(pred_claim_i.split('_')[1]), int(gold_claim_i.split('_')[1])
        # data_id, pred_claim_i, gold_claim_i

        precision_results[data_id][tuple([pred_claim_i, gold_claim_i])] = completion == 'yes' and math.exp(logprob) >= threshold

    # recall

    # if 'technontech2claimsip' in load_dir or 'text2ip' in load_dir:
    pattern = re.compile(r'^ips_openai_requests_\d+_recall\.jsonl\.gpt-4o-mini_response\.jsonl$')
    # else:
    #     pattern = re.compile(r'^openai_requests_\d+_recall\.jsonl\.gpt-4o-mini_response\.jsonl$')

    matched_files = [f for f in sorted(filenames) if pattern.match(f)]
    print(matched_files)

    responses = []
    for filename in matched_files:
        responses.extend(load_jsonl(os.path.join(load_dir, filename)))

    threshold = 0.9
    recall_results = defaultdict(dict)
    for i in range(len(responses)):
        line = json.loads(responses[i]['line'])
        custom_id = line['custom_id']
        try:
            completion = line['response']['body']['choices'][0]['logprobs']['content'][0]['token'].strip().lower()
            logprob = line['response']['body']['choices'][0]['logprobs']['content'][0]['logprob']

            data_id, pred_claim_i, gold_claim_i = custom_id.split('.')
            data_id, pred_claim_i, gold_claim_i = int(data_id.split('_')[1]), \
                int(pred_claim_i.split('_')[1]), int(gold_claim_i.split('_')[1])
            # data_id, pred_claim_i, gold_claim_i
            # print(completion, math.exp(logprob))
            # break
            recall_results[data_id][tuple([pred_claim_i, gold_claim_i])] = completion == 'yes' and math.exp(logprob) >= threshold
        except:
            print(i)


    # === Load your data/results ===
    # (Assuming load_paths, load_jsonl, train_dataset, new_data, precision_results, and recall_results are already defined)
    # For this example, we assume:
    #   - train_dataset['verifiable_claims'] is a list of lists of verifiable claims.
    #   - new_data is a list of dicts with key 'pred_claims'
    #   - precision_results and recall_results are dicts keyed by data instance id (integers)
    #   - load_dir is derived from one of the load_paths


    train_dataset, val_dataset, test_dataset = get_nsf_data_raw('..')

    # === Set up TF-IDF on verifiable claims ===

    # Flatten verifiable claims if needed (each inner list is from one data instance)
    investigation_proposals_flat = [claim for claims_list in train_dataset['investigation_proposals'] \
                              for claim in claims_list]

    # Fit TF-IDF vectorizer on verifiable claims only
    vectorizer = TfidfVectorizer()
    vectorizer.fit(investigation_proposals_flat)

    # === Compute Non-filtered Precision and Recall ===

    nonfiltered_precisions = []
    nonfiltered_recalls = []

    # Here, we assume that the keys of precision_results and recall_results correspond to the data instance indices.
    for di in range(len(new_data)):
        nonfiltered_precisions.append(compute_precision_from_results(precision_results[di]))
        nonfiltered_recalls.append(compute_recall_from_results(recall_results[di]))

    # === Redundancy Removal on Predicted Claims ===

    # For each data instance, we determine which predicted claims are unique.
    unique_pred_indices = {}  # data instance id -> list of indices of unique predicted claims

    for di in tqdm(range(len(new_data)), desc="Computing unique predicted claims"):
        pred_claims = new_data[di]['pred_ips']
        if len(pred_claims) == 0:
            unique_pred_indices[di] = []
            continue

        # Transform predicted claims using the vectorizer (fitted on verifiable claims)
        pred_tfidf = vectorizer.transform(pred_claims)

        # Compute pairwise cosine similarity among predicted claims
        similarity_matrix = cosine_similarity(pred_tfidf, pred_tfidf)
        np.fill_diagonal(similarity_matrix, 0)  # ignore self-similarity

        # Define a threshold above which two claims are considered redundant
        redundancy_threshold = 0.9

        unique_indices = []
        # For each claim, if it is too similar to any claim already selected, mark it as redundant.
        for i in range(len(pred_claims)):
            is_redundant = False
            for j in unique_indices:
                if similarity_matrix[i, j] >= redundancy_threshold:
                    is_redundant = True
                    break
            if not is_redundant:
                unique_indices.append(i)
        unique_pred_indices[di] = unique_indices

    # === Filter Precision and Recall Results Based on Unique Predicted Claims ===

    filtered_precision_results = {}
    filtered_recall_results = {}

    for di in unique_pred_indices:
        unique_set = set(unique_pred_indices[di])
        # Filter precision results: keep only keys where the predicted claim index is in the unique set.
        if di in precision_results:
            filtered_precision_results[di] = { key: val 
                for key, val in precision_results[di].items() if key[0] in unique_set }
            # print(list(unique_set)[0])

        else:
            filtered_precision_results[di] = {}

        if di in recall_results:
            filtered_recall_results[di] = { key: val 
                for key, val in recall_results[di].items() if key[0] in unique_set }
        else:
            filtered_recall_results[di] = {}

    # === Compute Filtered Precision and Recall ===

    filtered_precisions = []
    filtered_recalls = []

    for di in range(len(new_data)):
        filtered_precisions.append(compute_precision_from_results(filtered_precision_results[di]))
        filtered_recalls.append(compute_recall_from_results(filtered_recall_results[di]))

    # === Compute F-scores (optional) ===

    nonfiltered_fscores = [
        2 * (p * r) / (p + r) if (p + r) > 0 else 0.0 
        for p, r in zip(nonfiltered_precisions, nonfiltered_recalls)
    ]

    filtered_fscores = [
        2 * (p * r) / (p + r) if (p + r) > 0 else 0.0 
        for p, r in zip(filtered_precisions, filtered_recalls)
    ]

    # === Print Mean Metrics for Comparison ===

    print("Non-filtered Mean Precision:", np.mean(nonfiltered_precisions))
    print("Non-filtered Mean Recall:", np.mean(nonfiltered_recalls))
    print("Non-filtered Mean F-score:", np.mean(nonfiltered_fscores))

    print("Filtered Mean Precision:", np.mean(filtered_precisions))
    print("Filtered Mean Recall:", np.mean(filtered_recalls))
    print("Filtered Mean F-score:", np.mean(filtered_fscores))
    
    p_r_f_all[load_dirs[load_dir]]['precisions'] = filtered_precisions
    p_r_f_all[load_dirs[load_dir]]['recalls'] = filtered_recalls
    p_r_f_all[load_dirs[load_dir]]['fscores'] = filtered_fscores

    
results_path = f'../metrics/{task_name}_final_scores.json'
os.makedirs(os.path.dirname(results_path), exist_ok=True)
with open(results_path, 'wt') as output_file:
    json.dump(p_r_f_all, output_file, indent=4)

load_dir /shared_data0/weiqiuy/nsf-awards/results_gen_bsz1/technontech2ip/mistral-7b-instruct-v0.3_preds
['ips_openai_requests_11_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_1_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_3_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_5_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_7_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_9_recall.jsonl.gpt-4o-mini_response.jsonl']
Train dataset keys: dict_keys(['award_id', 'technical_abstract', 'non_technical_abstract', 'verifiable_claims', 'investigation_proposals', 'award_year', 'division', 'directorate', 'title', 'publications'])
Validation dataset keys: dict_keys(['award_id', 'technical_abstract', 'non_technical_abstract', 'verifiable_claims', 'investigation_proposals', 'award_year', 'division', 'directorate', 'title', 'publications'])
Test dataset keys: dict_keys(['award_id', 'technical_abstract', 'non_technical_abstrac

Computing unique predicted claims:   0%|          | 0/6000 [00:00<?, ?it/s]

Non-filtered Mean Precision: 0.6220692278381519
Non-filtered Mean Recall: 0.6367026312947172
Non-filtered Mean F-score: 0.5668663090728749
Filtered Mean Precision: 0.6221832809762764
Filtered Mean Recall: 0.6363571947867807
Filtered Mean F-score: 0.5668031970388508
load_dir /shared_data0/weiqiuy/nsf-awards/results_gen_bsz1/technontech2ip/lora_model_mistral-7b-instruct-v0.3.text2ip_instruct_user_assistant.r128_la64_lr1e-05_e3_s-1_wu100_bs2_schlinear_optadamw_8bit_emb_lm_preds
['ips_openai_requests_11_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_13_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_15_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_1_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_3_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_5_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_7_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_9_recall.jsonl.gpt-4o-mini_response.json

Computing unique predicted claims:   0%|          | 0/6000 [00:00<?, ?it/s]

Non-filtered Mean Precision: 0.6860650529109701
Non-filtered Mean Recall: 0.6510007175343863
Non-filtered Mean F-score: 0.6050153747778816
Filtered Mean Precision: 0.6866621306532877
Filtered Mean Recall: 0.6506722783809472
Filtered Mean F-score: 0.6053112920090958
load_dir /shared_data0/weiqiuy/nsf-awards/results_gen_bsz1/technontech2ip/lora_model_mistral-7b-instruct-v0.3.text2ip_instruct_user_assistant.matsci_and_20k.r128_la64_lr1e-05_e3_s-1_wu100_bs2_schlinear_optadamw_8bit_emb_lm_preds
['ips_openai_requests_11_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_13_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_15_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_1_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_3_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_5_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_7_recall.jsonl.gpt-4o-mini_response.jsonl', 'ips_openai_requests_9_recall.jsonl.gpt-4o-min

Computing unique predicted claims:   0%|          | 0/6000 [00:00<?, ?it/s]

Non-filtered Mean Precision: 0.7214339580132343
Non-filtered Mean Recall: 0.7363730668934035
Non-filtered Mean F-score: 0.7037837077163176
Filtered Mean Precision: 0.7219062320355083
Filtered Mean Recall: 0.7358762414965782
Filtered Mean F-score: 0.7039141995800923


In [12]:
import numpy as np

for key in p_r_f_all:
    print(key, np.mean(p_r_f_all[key]['precisions']))

Mistral 0.6217010028713976
Mistral-ft 0.7351236364577115
Qwen 0.4425878427128427
Qwen-ft 0.7244910867727045
Mistral-comb 0.5713429223554224
Mistral-comb-ft 0.7397432012840102
Qwen-comb 0.5493993839493839
Qwen-comb-ft 0.7016413007292844


In [13]:
with open(results_path, 'rt') as input_file:
    temp_data = json.load(input_file)

for key in temp_data:
    print(key, np.mean(temp_data[key]['precisions']))

Mistral 0.6217010028713976
Mistral-ft 0.7351236364577115
Qwen 0.4425878427128427
Qwen-ft 0.7244910867727045
Mistral-comb 0.5713429223554224
Mistral-comb-ft 0.7397432012840102
Qwen-comb 0.5493993839493839
Qwen-comb-ft 0.7016413007292844


In [14]:
results_path

'../metrics/technontech2ip_scores.json'

In [9]:
line

{'id': 'batch_req_67cce310d6148190bd1c4be22d4fbdc1',
 'custom_id': 'data_422.pred_8.gold_3',
 'response': {'status_code': 200,
  'request_id': 'd4a1fc11e65ce85c46fd100943c925ae',
  'body': {'error': {'message': 'Timed out generating response. Please try again with a shorter prompt or with `max_tokens` set to a lower value.',
    'type': 'internal_error',
    'param': None,
    'code': 'request_timeout'}}},
 'error': None}